# Detection Module

    The main goal of the detection module is to use the gazetteers out of the ontologies used to enrich PropaPhen into PropaPhen+ to discover relationships between network nodes/systems and the gufo:Entities by text.

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Libraries

### Installing

In [5]:
#!pip install pandas
#!pip install tqdm
#!pip install nltk
#!pip install gatenlp
#!pip install py4j
#!pip install pyodide
#!pip install ipywidgets
#!pip install neo4j

### Standard

In [7]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
import glob

In [8]:
from gatenlp import Document
from gatenlp.gateworker import GateWorker

### Custom libraries

In [9]:
import sys
sys.path.append('lib/')

In [10]:
from detection.relationshipextraction import RelationshipDiscovery, GateExtractor, CleanDicts, rmToRelationCSV
from detection.schema import Term, Concept, df_to_concepts, cleaningPlaceStr, conceptsToGazetteer
from detection.worldumls import umlsConceptCleanner, isEnglish, worldConceptCleanner
from detection.worldumls import ClearnWorldKGGazetteer
#import detection.observationclustering

## Globals

In [11]:
path_to_covid_journals = "data/textual/covid/newspaper/"
path_to_kb_gazetteer = '../data/gazetteers/kbgazetteer.csv'
path_to_netwoork_gazetteer = '../data/gazetteers/world_gazetteer_en.csv'
path_to_lsts = "data/lst/"
path_to_relation_folder = "../data/neo4j/"
path_to_observationcsv = "../data/neo4j/observations.csv"

## Relationship Discovery

### KB Gazetteers

In [ ]:
kb_concept_list = []
network_concept_list = []

In [ ]:
df_kb = pd.read_csv(path_to_kb_gazetteer)

In [ ]:
df_kb.head()

In [ ]:
kb_concept_list = df_to_concepts(df_kb)

In [ ]:
for i in tqdm(range(len(kb_concept_list))):
    kb_concept_list[i] = umlsConceptCleanner(kb_concept_list[i])
    kb_concept_list[i] = umlsConceptCleanner(kb_concept_list[i])

In [ ]:
umlsdict = conceptsToGazetteer(kb_concept_list,path_to_lsts+"umls.lst",cleaningPlaceStr)

### Place Gazetteers

In [ ]:
df_network = pd.read_csv(path_to_netwoork_gazetteer)

In [ ]:
clear_net_list = ['"Nga"']

In [ ]:
df_network = ClearnWorldKGGazetteer(df_network,clear_net_list)

In [ ]:
df_network.head()

In [ ]:
network_concept_list = df_to_concepts(df_network)

In [ ]:
# Pre-processing network
#for i in tqdm(range(len(network_concept_list))):
#    network_concept_list[i] = worldConceptCleanner(network_concept_list[i])

In [ ]:
# Normal
print("Usual name")
normalplacesdict = conceptsToGazetteer(network_concept_list,path_to_lsts+"places.lst",cleaningPlaceStr)
# Cap
#print("Cap name")
#capdict = conceptsToGazetteer(network_concept_list,path_to_lsts+"places_cap.lst",capPlaceStr)
# Lower
#print("Lower name")
#lowerdict = conceptsToGazetteer(network_concept_list,path_to_lsts+"places_lower.lst",lowerPlaceStr)

### GATE

In [ ]:
gs = GateWorker(start=False, auth_token="1234")

In [ ]:
normalplacesdict, umlsdict = CleanDicts(normalplacesdict, umlsdict)

In [ ]:
gateExtractor = GateExtractor(umlsdict,normalplacesdict)

In [ ]:
# Annie
gs.worker.loadMavenPlugin("uk.ac.gate.plugins", "annie", "8.6")
# now load the prepared ANNIE pipeline from the plugin
pipeline = gs.worker.loadPipelineFromPlugin("uk.ac.gate.plugins","annie", "/resources/ANNIE_with_defaults.gapp")
pipeline.getName()

In [ ]:
gateExtractor.extra_pr['annie'] = pipeline

## Medical Articles Relationship Discovery

In [ ]:
corpus = gs.getCorpus4Name('PreDiViD-CORD19-2019-12')

In [ ]:
rd = RelationshipDiscovery(corpus, gateExtractor,gs)

In [ ]:
rmDoc = rd.rmGen.directTermMatching('PreDiViD-CORD19-Abstract-2019-12-Doc')

In [ ]:
rmParagraph = rd.rmGen.paragraphTermMatching('PreDiViD-CORD19-Abstract-2019-12-Paragraph')

In [ ]:
rmSentence = rd.rmGen.sentenceTermMatching('PreDiViD-CORD19-Abstract-2019-12-Sentence')

In [ ]:
df_rm = rmToRelationCSV(rmDoc, 'Medical', 1, 'hasPresence') 
df_rm.to_csv(path_to_relation_folder+rmDoc.matrix_id+".csv", index=False)
df_rmParagraph = rmToRelationCSV(rmParagraph, 'Medical', 1, 'hasPresence') 
df_rmParagraph.to_csv(path_to_relation_folder+rmParagraph.matrix_id+".csv", index=False)
df_rmSentence = rmToRelationCSV(rmSentence, 'Medical', 1, 'hasPresence') 
df_rmSentence.to_csv(path_to_relation_folder+rmSentence.matrix_id+".csv", index=False)

## Online Newspaper Relationship Discovery

In [ ]:
corpus = gs.getCorpus4Name('PreDiViD')
rd = RelationshipDiscovery(corpus, gateExtractor,gs)
rmDoc = rd.rmGen.directTermMatching('PreDiViD-Aylien-2019-11-Doc')
rmParagraph = rd.rmGen.paragraphTermMatching('PreDiViD-Aylien-2019-11-Paragraph')
rmSentence = rd.rmGen.sentenceTermMatching('PreDiViD-Aylien-2019-11-Sentence')
df_rm = rmToRelationCSV(rmDoc, 'Journal', 1, 'hasPresence') 
df_rm.to_csv(path_to_relation_folder+rmDoc.matrix_id+".csv", index=False)
df_rmParagraph = rmToRelationCSV(rmParagraph, 'Journal', 1, 'hasPresence') 
df_rmParagraph.to_csv(path_to_relation_folder+rmParagraph.matrix_id+".csv", index=False)
df_rmSentence = rmToRelationCSV(rmSentence, 'Journal', 1, 'hasPresence') 
df_rmSentence.to_csv(path_to_relation_folder+rmSentence.matrix_id+".csv", index=False)

### Observation Mining

In [2]:
from lib.kgce.schema.semantic.neo4jclasses import Neo4jRelation
from lib.kgce.neo4j.handler import Neo4jWrapper

In [3]:
from neo4j import GraphDatabase
from tqdm import tqdm


class Neo4jWrapper:

    def __init__(self, uri, userName, password):
        self.uri = uri
        self.userName = userName
        self.password = password
        # Connect to the neo4j database server
        self.graphDB_Driver  = GraphDatabase.driver(uri, auth=(userName, password)) 
        
    def sendQuery(self, cql_commands):
        result = []
        done_queries = []
        with self.graphDB_Driver.session() as graphDB_Session:
            for cqlCreate in tqdm(cql_commands):
                try:
                    result += [graphDB_Session.run(cqlCreate).to_df()]
                    done_queries.append(cqlCreate)
                except Exception as e:
                    tqdm.write(str(e))
                    tqdm.write(cqlCreate)
                    result += [str(e)]
        return result
    
    def closeConnection(self):
        self.graphDB_Driver.close()

In [4]:
neowrapper = Neo4jWrapper(uri="bolt://localhost:7687",userName="neo4j",password="test")

In [65]:
result = neowrapper.sendQuery([
    """MATCH (n:Country)<-[r:hasPresence]-(c) 
    WHERE toInteger(r.intensity) >= 100
    RETURN n.wkgs_nameEn as System_Name, n.id, c.name, c.id, r.intensity as intensity;"""
])

100%|█████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.75it/s]


In [109]:
const_max_path = 3263433+2 # All Node from CUI + 2 for the leaf AUI nodes
def Theta(nodeid1, nodeid2,dictPath,neowrapper):
    # If calculated return
    if (nodeid1, nodeid2) in dictPath:
        return dictPath[(nodeid1, nodeid2)],dictPath
    elif nodeid1 == nodeid2:
        dictPath[(nodeid1, nodeid2)] = 0
        return dictPath[(nodeid1, nodeid2)], dictPath
    else:
        result = neowrapper.sendQuery([
            """MATCH (c1:UMLS|id:"{0}"\),(c2:UMLS|id:"{1}"\), 
            p = shortestPath((c1)-[*]-(c2))
            RETURN length(p) as value""".format(
            nodeid1,nodeid2).replace(
            "|","{").replace("\\","}")
        ])
        value = const_max_path
        if(not result[0].empty):
            value = int(result[0]['value'].iloc[0])
        dictPath[(nodeid1, nodeid2)] = value
        return value, dictPath

In [67]:
df_result = result[0].groupby(['System_Name','n.id'],as_index=False).agg(list)

In [68]:
df_result

,System_Name,n.id,c.name,c.id,intensity
0,"""Angola""",wkg:424310875,"[code, PCR, Description, 0: Eye problem(s) had...","[A18625219, A25746597, A7734618, A33693090, A2...","[1104, 112, 168, 144, 1016, 172, 1040, 416, 11..."
1,"""Argentina""",wkg:249399280,"[In, In, electrocardiogram: 1:1 atrioventricul...","[A3144515, A12807184, A17276557, A18589955, A1...","[136, 136, 106, 120, 104, 104, 164, 104, 136, ..."
2,"""Australia""",wkg:424315584,"[MK, code, MK, Information, contract, MK, cont...","[A19287824, A18625219, A24583083, A8317364, A8...","[508, 552, 508, 208, 104, 508, 104, 208, 508, ..."
3,"""Belarus""",wkg:249399300,"[virus, human, agent, Stage B: infection, O/E:...","[C0319157, A18669849, C1551364, A32476065, A48...","[272, 136, 152, 184, 272, 160, 176, 204, 240, ..."
4,"""Belgium""",wkg:1684793666,"[In, O/E: E.M. micr.: virus, electrocardiogram...","[A3144515, A22865475, A17276557, C1706281, A18...","[213, 281, 522, 143, 144, 100, 208, 119, 184, ..."
5,"""Bulgaria""",wkg:424315709,[CDISC ADAS-Cog - Naming Objects and Fingers: ...,"[A21398167, A9333675, A25726660, A17148156, C1...","[616, 280, 140, 350, 168, 322, 476, 112, 588, ..."
6,"""Canada""",wkg:424313760,"[contract, MK, MK, MK, Information, electrocar...","[A18618609, A24367753, A24370628, A24583083, C...","[104, 508, 508, 508, 208, 133, 208, 508, 208, ..."
7,"""China""",wkg:424313582,"[EBV, Coronavirus, RSV, Information, Stage B: ...","[A26636222, A32453564, A2876241, C1561527, A32...","[160, 178, 188, 312, 468, 344, 144, 168, 229, ..."
8,"""Denmark""",wkg:432424968,"[IV:19, II:23, 6:2 FTAB 6:2 fluorotelomer sulf...","[C5761443, C5761581, A33643234, A33643108, A10...","[136, 1040, 176, 120, 416, 160, 168, 208, 1016..."
9,"""France""",wkg:1363947712,"[Bordetella, infection, Place, E, apparatus, M...","[C0006013, A4386825, A7567961, A3122833, A8396...","[128, 171, 704, 206, 128, 128, 484, 169, 104, ..."


In [101]:
country_measure = []
dictPath = {}

In [111]:
for index, row in df_result.iterrows():
    print(row['System_Name'])
    # For each country
    for umlsel in row['c.id']:
        # For each umls entity
        theta,dictPath = Theta(umlsel,'C5203670',dictPath,neowrapper)

"Angola"


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:12<00:00, 12.18s/it]


"Argentina"


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:13<00:00, 13.14s/it]


"Australia"
"Belarus"


...

100%|█████████████████████████████████████████████████████████████████| 1/1 [00:12<00:00, 12.73s/it]


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:12<00:00, 12.53s/it]


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:12<00:00, 12.53s/it]


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:12<00:00, 12.58s/it]


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:12<00:00, 12.56s/it]


In [132]:
dictPresence = {}
for index, row in df_result.iterrows():
    presenceValue = 0
    thetaSum = 1
    # For each country
    for umlsel in row['c.id']:
        # For each umls entity
        thetaSum += dictPath[umlsel,'C5203670']
    dictPresence[row['System_Name']] = len(row['c.id'])/thetaSum

In [133]:
dictPresence

{'"Angola"': 0.3357142857142857,
 '"Argentina"': 0.3291139240506329,
 '"Australia"': 0.3333333333333333,
 '"Belarus"': 0.330188679245283,
 '"Belgium"': 0.3362369337979094,
 '"Bulgaria"': 0.3353221957040573,
 '"Canada"': 0.3333333333333333,
 '"China"': 0.33516483516483514,
 '"Denmark"': 0.3357664233576642,
 '"France"': 0.0002832045900575288,
 '"Germany"': 0.3351063829787234,
 '"Grenada"': 0.00020579206671582812,
 '"Haiti"': 0.33554817275747506,
 '"India"': 0.33415233415233414,
 '"Iran"': 0.3,
 '"Kazakhstan"': 0.3344103392568659,
 '"Kyrgyzstan"': 0.330188679245283,
 '"Luxembourg"': 0.3357664233576642,
 '"Netherlands"': 0.3359746434231379,
 '"Portugal"': 0.0002076283513757675,
 '"Romania"': 0.3351063829787234,
 '"Russia"': 0.0004755100104954912,
 '"Saudi Arabia"': 0.3269230769230769,
 '"Sierra Leone"': 0.325,
 '"Singapore"': 0.32,
 '"Spain"': 0.33636363636363636,
 '"Switzerland"': 0.325,
 '"Taiwan"': 0.330188679245283,
 '"Turkey"': 0.330188679245283,
 '"United Kingdom"': 0.333333333333333

In [ ]:
df_result.to_csv("data/csv/observations_phrase.csv", index=False)  

In [9]:
result = neowrapper.sendQuery([
    """MATCH (n:Country)<-[r:hasPresence]-(c) 
    WHERE toInteger(r.intensity) >= 1000 AND r.source = "Journal"
    RETURN n.wkgs_nameEn as System_Name, n.id, c.name, c.id, r.intensity as intensity;"""
])

100%|█████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.69it/s]


In [10]:
df_result = result[0].groupby(['System_Name','n.id'],as_index=False).agg(list)

In [11]:
df_result

,System_Name,n.id,c.name,c.id,intensity
0,"""Angola""",wkg:424310875,"[code, MK, II:23, MK, MK, MK, MK, II:23, MK, c...","[A18625219, A20722030, A34717172, A24370628, A...","[1104, 1016, 1040, 1016, 1016, 1016, 1016, 104..."
1,"""Belgium""",wkg:1684793666,"[MK, MK, MK, II:23, MK, code, code, MK, MK, MK...","[A24367753, A32664340, A24370628, A34717172, A...","[1016, 1016, 1016, 1040, 1016, 1104, 1104, 101..."
2,"""Bulgaria""",wkg:424315709,[stress test electrocardiogram: 2:1 atrioventr...,"[A17248370, A20722030, A7565400, A32664340, C1...","[4592, 3556, 1456, 3556, 1456, 3864, 1456, 355..."
3,"""China""",wkg:424313582,[stress test electrocardiogram: 2:1 atrioventr...,[A17248370],[1200]
4,"""Denmark""",wkg:432424968,"[II:23, MK, MK, MK, MK, II:23, MK, MK, code, s...","[C5761581, A16758859, A32664337, A20759385, C5...","[1040, 1016, 1016, 1016, 1016, 1040, 1016, 101..."
5,"""France""",wkg:1363947712,"[O/E: E.M. micr.: virus, Description, 6:2 FTAB...","[A22865475, A32797912, A33643234, C3639183, A1...","[1447, 1344, 1456, 1324, 8832, 1024, 1136, 146..."
6,"""Germany""",wkg:1683325355,"[MK, MK, MK, MK, code, MK, stress test electro...","[A32664340, A16758859, A24583083, A24370628, A...","[1524, 1524, 1524, 1524, 1656, 1524, 1968, 152..."
7,"""Grenada""",wkg:424316074,"[Description, electrocardiogram: 1:1 atriovent...","[A8317986, A17276557, C5761581, A8317350, A227...","[1680, 4160, 10400, 2080, 1040, 1120, 1000, 17..."
8,"""Haiti""",wkg:424297281,"[MK, code, MK, code, MK, MK, MK, MK, II:23, st...","[A32664337, A18625219, A24367753, A18553518, C...","[1016, 1104, 1016, 1104, 1016, 1016, 1016, 101..."
9,"""Kazakhstan""",wkg:424311521,"[virus, virus, virus, virus, virus, O/E: E.M. ...","[C0319157, A18650525, A9333675, A4387104, A186...","[1088, 1088, 1088, 1088, 1088, 1088, 1088, 108..."
